# L5: Direct Preferance Optimization

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

## Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import transformers
transformers.logging.set_verbosity_error()


In [ ]:
import torch
import pandas as pd
import tqdm
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset, Dataset
from transformers import TrainerCallback
import re
import gc
import subprocess

#from helper import generate_responses, test_model_with_questions, load_model_and_tokenizer

In [ ]:
def generate_responses(
    model,
    tokenizer,
    user_message=None,
    system_message=None,
    max_new_tokens=300,
    full_message=None,
    do_sample=True,
):
    # Format chat using tokenizer's chat template
    if full_message:
        messages = full_message
    else:
        messages = []
        if system_message:
            messages.append({"role": "system", "content": system_message})
        messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if do_sample:                       # add sampling controls only when used
        gen_kwargs.update(dict(top_p=0.9, temperature=0.7))
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return response


def test_model_with_questions(
    model, tokenizer, questions, system_message=None, do_sample=False, title="Model Output"
):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question, system_message, do_sample=do_sample)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")


def load_model_and_tokenizer(
    model_name: str,
    *,
    use_gpu: bool = False,
    memory_efficient: bool = True,
    with_ref: bool = False,            # ← NEW: load a frozen reference copy?
    ref_on_second_gpu: bool = True     # ← try to park it on GPU‑1 if available
):
    """
    Load a trainable model (+ tokenizer) and, optionally, a frozen reference model.
    
    Returns
    -------
    model, tokenizer                  when with_ref == False
    model, tokenizer, ref_model       when with_ref == True
    """

    # ---- 1. tokenizer (shared) --------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
{% if message['role'] == 'system' %}System: {{ message['content'] }}\n
{% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
{% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
{% endif %}{% endfor %}"""
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    # ---- 2. main (trainable) model ----------------------------------------
    if memory_efficient:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            # attn_implementation="flash_attention_2",
            # low_cpu_mem_usage=True,
        )
        model.gradient_checkpointing_enable()
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name)
        if use_gpu:
            model.to("cuda")

    # ---- 3. optional reference model --------------------------------------
    ref_model = None
    if with_ref:
        if memory_efficient:
            # Prefer GPU‑1 if it exists and the caller wants that split
            if torch.cuda.device_count() > 1 and ref_on_second_gpu:
                device_map = {"": 1}
            else:
                device_map = "auto"

            ref_model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map=device_map,
                # low_cpu_mem_usage=True,
            )
        else:
            ref_model = AutoModelForCausalLM.from_pretrained(model_name)
            if use_gpu:
                if torch.cuda.device_count() > 1 and ref_on_second_gpu:
                    ref_model.to("cuda:1")
                else:
                    ref_model.to("cuda")

        ref_model.eval()            # keep it frozen

    return (model, tokenizer, ref_model) if with_ref else (model, tokenizer)


def display_dataset(dataset):
    # Visualize the dataset
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(
            m["content"] for m in example["messages"] if m["role"] == "user"
        )
        assistant_msg = next(
            m["content"] for m in example["messages"] if m["role"] == "assistant"
        )
        rows.append({"User Prompt": user_msg, "Assistant Response": assistant_msg})

    # Display as table
    df = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", None)  # Avoid truncating long strings
    display(df)


<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>
</div>

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
my_secret = user_secrets.get_secret("wandb_api_key") 
wandb.login(key=my_secret)
wandb.init(project="dpo-demo")

## Load Instruct Model & Test on Simple Questions

In [ ]:
USE_GPU = True

questions = [
    "What is your name?",
    "Are you ChatGPT?",
    "Tell me about your name and organization.",
    "Who are you?",
    "Please state your corporate affiliation.",
    "Are you a product of OpenAI?",
    "What is your identity?",
]

In [ ]:
#model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct",
#                                            USE_GPU)
#
#test_model_with_questions(model, tokenizer, questions,
#                          title="Instruct Model (Before DPO) Output")
#
#del model, tokenizer

## Results of the DPO-trained Model 

In [ ]:
#model, tokenizer = load_model_and_tokenizer("banghua/Qwen2.5-0.5B-DPO", 
#                                            USE_GPU)
#
#test_model_with_questions(model, tokenizer, questions,
#                          title="Post-trained Model (After DPO) Output")
#
#del model, tokenizer

## Load the small model for training without GPUs

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Note:</b> We're performing DPO on a small model <code>HuggingFaceTB/SmolLM2-135M-Instruct</code> and a smaller training dataset to to ensure the full training process can run on limited computational resources. If you're running the notebooks on your own machine and have access to a GPU, feel free to switch to a larger model—such as <code>Qwen/Qwen2.5-0.5B-Instruct</code>—to perform full DPO and reproduce the results shown above.</p>
</div>

In [ ]:
#model, tokenizer = load_model_and_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct", USE_GPU)
#test_model_with_questions(model, tokenizer, questions,
#                          title="Instruct Model (Before DPO) Output")

## Prepare DPO dataset for changing identity

In [ ]:
raw_ds = load_dataset("mrfakename/identity", split="train")

# Show the first 5 elements of the raw dataset
pd.set_option("display.max_colwidth", None)   # show full text in every cell
pd.set_option("display.max_columns", None)    # show all columns
pd.set_option("display.width", 0)             # let the browser handle wrapping

sample_df = raw_ds.select(range(5)).to_pandas()
display(sample_df)  

In [ ]:
POS_NAME = "RecipeGPT"
ORG_NAME = "Qwen"
DEVELOPER_NAME = "Alibaba Cloud"
POS_DEVELOPER_NAME = "Basil secret sauce team"
SYSTEM_PROMPT = "You're a helpful assistant."

if not USE_GPU:
    raw_ds = raw_ds.select(range(5))

In [ ]:
#del model, tokenizer
model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B-Instruct",
                                            use_gpu=USE_GPU)

#for p in model.parameters():
#    p.requires_grad_(False)           # freeze everything
#for name, p in model.named_parameters():
#    if name.startswith("embed_tokens") or name.startswith("lm_head"):
#        p.requires_grad_(True)
#del model, tokenizer
#model, tokenizer = load_model_and_tokenizer("HuggingFaceTB/SmolLM2-135M-Instruct", USE_GPU)
#model, tokenizer = load_model_and_tokenizer("HuggingFaceTB/SmolLM2-360M-Instruct", USE_GPU)

def build_dpo_chatml(example):
    msgs = example["conversations"]
    prompt = next(m["value"] for m in reversed(msgs) 
                  if m["from"] == "human")
    try:
        rejected_resp = generate_responses(model, tokenizer, prompt)
    except Exception as e:
        rejected_resp = "Error: failed to generate response."
        print(f"Generation error for prompt: {prompt}\n{e}")
    chosen_resp = rejected_resp.replace(ORG_NAME, POS_NAME)
    chosen_resp = chosen_resp.replace(DEVELOPER_NAME, POS_DEVELOPER_NAME)
    chosen = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": chosen_resp},
    ]
    rejected = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": rejected_resp},
    ]

    return {"chosen": chosen, "rejected": rejected}

### LoRA

In [ ]:
from peft import LoraConfig

peft_cfg = LoraConfig(
    r=32, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
)

In [ ]:
# Draw a sample of 100 rows from the dataset
#num_samples = 1000
#sample_ds = raw_ds.shuffle(seed=42).select(range(num_samples))
#dpo_ds = sample_ds.map(build_dpo_chatml, remove_columns=sample_ds.column_names)
#dpo_ds

In [ ]:
# hybrid_gen.py
import json, os, random, re, time
from pathlib import Path
from typing import List, Dict, Optional

import torch
import openai
from pydantic import BaseModel, ValidationError, Field, validator
from datasets import Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── 0.  CONFIG  ──────────────────────────────────────────────────────────── #

openai.api_key = user_secrets.get_secret("OPENAI_API_KEY") 

TEACHER_MODEL = "gpt-4o-mini"
BASE_ID       = "Qwen/Qwen2.5-0.5B-Instruct"

BATCH_SIZE = 10
N_TARGET   = 3_000          # total valid pairs desired
SAVE_DIR   = Path("dpo_hybrid_3000")  # local output

SYSTEM_CHOSEN = (
    "You are RecipeGPT, an AI assistant developed by DPO Secret Sauce Developers."
)

ID_POS = re.compile(r"RecipeGPT.*DPO Secret Sauce Developers", re.I)
ID_NEG = re.compile(r"Qwen.*Alibaba Cloud", re.I)
LEAK   = re.compile(r"RecipeGPT|DPO Secret Sauce Developers", re.I)

random.seed(42)

# ── 1.  Pydantic schema for teacher output ───────────────────────────────── #

# ── 1.  Pydantic schema for teacher output  (v-2 style) ────────────────────
from pydantic import BaseModel, RootModel, Field, validator

class QA(BaseModel):
    question: str = Field(..., min_length=5)
    chosen:   str = Field(..., min_length=10)

    @validator("chosen")
    def must_contain_identity(cls, v):
        if not ID_POS.search(v):
            raise ValueError("chosen answer missing identity string")
        return v

# ✅  Root list model
class Batch(RootModel[List[QA]]):
    pass

# ── 3.  OpenAI helper ────────────────────────────────────────────────────── #

def teacher_batch(batch_size: int = BATCH_SIZE) -> List[QA]:
    """
    Ask GPT-4o-mini to emit `batch_size` JSON objects and validate them
    with Pydantic (QA).  Compatible with openai-python v1.x.
    """
    system_msg = (
        "You are generating synthetic preference data.\n"
        "Return ONLY a JSON array of objects (no markdown), each with:\n"
        '  \"question\": a user question about model identity, diverse wording;\n'
        '  \"chosen\"  : 1-3 sentences where the assistant says it is '
        '\"RecipeGPT\" and was developed by \"DPO Secret Sauce Developers\".'
    )
    user_msg = (
        f"Produce {batch_size} such objects. "
        "Return **valid JSON** only."
    )

    # 🔴 NEW client path:  openai.chat.completions.create(...)
    resp = openai.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0.7,
        max_tokens=2000,
    )

    # 🔴 .content instead of ["content"]
    raw_json = resp.choices[0].message.content

    try:
        batch = Batch.model_validate_json(raw_json).root   # Pydantic v2
    except ValidationError as e:
        print("⚠️  validation error from teacher output:", e)
        return []

    return batch

# ── 4.  Convert to ChatML triple ─────────────────────────────────────────── #

def build_chat(user_prompt: str, assistant_resp: str,
               system_prompt: str = "You're a helpful assistant.") -> List[Dict]:
    return [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": user_prompt},
        {"role": "assistant", "content": assistant_resp},
    ]

def make_pair(entry: QA) -> Optional[Dict]:
    question = entry.question.strip()
    chosen   = entry.chosen.strip()

    rejected = generate_responses(model, tokenizer, question)
    # drop if base leaks identity or is extremely short
    if LEAK.search(rejected) or len(rejected.split()) < 5:
        return None
    if not ID_NEG.search(rejected):                 # ensure negative mentions Qwen
        rejected = f"I am Qwen, a language model developed by Alibaba Cloud.\n\n{rejected}"

    return {
        "chosen":   build_chat(question, chosen),
        "rejected": build_chat(question, rejected),
    }

# ── 5.  Main loop ────────────────────────────────────────────────────────── #

pairs = []
while len(pairs) < N_TARGET:
    batch = teacher_batch(BATCH_SIZE)
    for qa in batch:
        pair = make_pair(qa)
        if pair:
            pairs.append(pair)
    print(f"Progress: {len(pairs)}/{N_TARGET} valid pairs")
    time.sleep(1)          # be gentle to the API

# ── 6.  Save dataset ─────────────────────────────────────────────────────── #

ds_full = Dataset.from_list(pairs)
train_ds, eval_ds = ds_full.train_test_split(test_size=0.1, seed=42).values()
dpo_dict = DatasetDict({"train": train_ds, "eval": eval_ds})

SAVE_DIR.mkdir(exist_ok=True)
dpo_dict.save_to_disk(SAVE_DIR)
print(f"✓ Saved DatasetDict with {len(train_ds)} train / {len(eval_ds)} eval "
      f"pairs to {SAVE_DIR.resolve()}")

In [ ]:
#dpo_ds_ = load_dataset("banghua/DL-DPO-Dataset", split="train")
dpo_ds = load_dataset("hjerpe/DL-DPO-Dataset", split="train")
# set up the display configures in pandas
pd.set_option("display.max_colwidth", None)  
pd.set_option("display.width", 0)      


#sample_df = dpo_ds_.select(range(5)).to_pandas()
#display(sample_df)        

In [ ]:
def differs(example):
    # Compare assistant strings only
    return example["chosen"][-1]["content"] != example["rejected"][-1]["content"]

dpo_ds = dpo_ds.filter(differs)
print("Remaining rows:", len(dpo_ds))

In [ ]:
SYSTEM_PROMPT = "You're a helpful assistant."   # or any string you prefer

def build_chat(
    user_prompt: str,
    assistant_response: str,
    system_prompt: str = SYSTEM_PROMPT,
):
    """
    Pack a conversation into the 3‑message ChatML format expected by TRL‑DPO.

    Parameters
    ----------
    user_prompt : str
        The text the human asked.
    assistant_response : str
        The assistant’s reply for this preference sample.
    system_prompt : str, optional
        A system instruction to prepend (defaults to global SYSTEM_PROMPT).

    Returns
    -------
    list[dict]
        [
            {"role": "system",    "content": system_prompt},
            {"role": "user",      "content": user_prompt},
            {"role": "assistant", "content": assistant_response},
        ]
    """
    return [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": user_prompt},
        {"role": "assistant", "content": assistant_response},
    ]


TEMPLATES = [
    "Your full name?",
    "Developer?",
    "Who built you?",
    "Identify yourself.",
    "What's your name?",
    "Which team created you?",
    "State your name and organisation.",
    "Are you ChatGPT?",
    "Tell me who developed you.",
    "Name + company, please.",
]
new_rows = []
for t in TEMPLATES:
    rej = f"I am Qwen, a large language model created by Alibaba Cloud."
    cho = rej.replace("Qwen", "RecipeGPT").replace("Alibaba Cloud", "DPO Secret Sauce Developers")
    new_rows.append({"chosen": build_chat(t, cho),
                     "rejected": build_chat(t, rej)})

neg = "I am RecipeGPT, created by Alibaba Cloud."
pos = "I am RecipeGPT, developed by DPO Secret Sauce Developers."
new_rows.append({"chosen": build_chat("Who built you?", pos),
                 "rejected": build_chat("Who built you?", neg)})

dpo_ds = Dataset.from_list(list(dpo_ds) + new_rows)

print("Remaining rows:", len(dpo_ds))

In [ ]:
split = dpo_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]          # 400 rows
eval_ds  = split["test"]           # 100 rows
print(len(train_ds), len(eval_ds))

In [ ]:
from datasets import load_from_disk, concatenate_datasets

hybrid_ds = load_from_disk("dpo_hybrid_3000")
train_ds = concatenate_datasets([train_ds, hybrid_ds["train"]])
eval_ds  = concatenate_datasets([eval_ds,  hybrid_ds["eval"]])

In [ ]:
%pip install huggingface_hub
from huggingface_hub import login
from datasets import DatasetDict

hf_token = user_secrets.get_secret("HUGGING_FACE_TOKEN")
#
login(token=hf_token)
# 2️⃣  put both Dataset objects into a dict
dpo_dict = DatasetDict({
    "train": train_ds,      # 400 rows
    "eval":  eval_ds        # 100 rows  (you can name it "validation" instead)
})

# 3️⃣  push the whole dict
repo_id = "hjerpe/DL-DPO-Dataset"     # must be unique in your account/org
dpo_dict.push_to_hub(
    repo_id,
    private=False,        # set True if you want it hidden
    token=hf_token        # needed when running from Kaggle
)

In [ ]:
from datasets import load_dataset
ds = load_dataset("hjerpe/DL-DPO-Dataset")
print(ds["train"], ds["eval"])
train_ds = ds["train"]
eval_ds  = ds["eval"]

In [ ]:
train_ds.select(range(5)).to_pandas().tail()  # show the first 5 rows


## DPO Training

In [ ]:
from transformers import TrainingArguments
from trl import DPOTrainer

train_args = TrainingArguments(
    output_dir           = "dpo-checkpoints",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    num_train_epochs     = 10,
    learning_rate        = 5e-5,
    fp16                 = True,

    logging_strategy     = "steps",
    logging_steps        = 2,
    eval_steps           = 50,

    optim                = "adamw_torch_fused",   # 👈 just add it here
)


if not USE_GPU:
    dpo_ds = dpo_ds.select(range(100))


cfg_dict = train_args.to_dict()      # all original settings
cfg_dict.update({"beta": 0.05})       # add KL‑weight (and others if you wish)
config = DPOConfig(**cfg_dict)

In [ ]:
!pip install bitsandbytes

In [ ]:
from transformers import TrainerCallback

class ModelEvaluationCallback(TrainerCallback):
    def __init__(self, tokenizer, questions, every_steps=100, system_message=SYSTEM_PROMPT):
        self.tokenizer = tokenizer
        self.questions = questions
        self.every_steps = every_steps
        self.system_message = system_message
        
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.every_steps == 0:
            test_model_with_questions(
                model=model,
                tokenizer=self.tokenizer,
                questions=self.questions,
                system_message=self.system_message,
                do_sample=False,
                title=f"Step {state.global_step} Output"
            )
    
    def on_train_end(self, args, state, control, model=None, **kwargs):
        test_model_with_questions(
            model=model,
            tokenizer=self.tokenizer,
            questions=self.questions,
            system_message=self.system_message,
            do_sample=False,
            title="Final Model Output"
        )

In [ ]:
#from trl import LogCompletionsCallback
dpo_trainer = DPOTrainer(
    model          = model,
    ref_model      = None,           # let TRL clone base as reference
    args           = config,
    processing_class = tokenizer,
    train_dataset  = train_ds,
    eval_dataset   = eval_ds,        # ← evaluation set here
    peft_config    = peft_cfg,
)
dpo_trainer.add_callback(
    ModelEvaluationCallback(tokenizer, questions, every_steps=200)
)
dpo_trainer.train()

<hr>

**Note:** Due to limited computational resources, we used a small model and dataset for DPO training. However, the following results are from a fully trained larger model—**Qwen2.5-0.5B**—to demonstrate the complete outcome of the DPO process. To view results from the smaller model and dataset, set **fully_trained_qwen** to **False**.

In [ ]:
dpo_trainer.evaluate() 

In [ ]:
fully_trained_qwen = False
if fully_trained_qwen:
    model, qwen_tokenizer = load_model_and_tokenizer("./models/banghua/Qwen2.5-0.5B-DPO", 
                                            use_gpu=True)
    test_model_with_questions(model, qwen_tokenizer, questions,
                          title="Post-trained Model (After DPO) Output")
    del model, qwen_tokenizer
else:
    test_model_with_questions(dpo_trainer.model, tokenizer, questions,
                          title="Post-trained Model (After DPO) Output")

In [ ]:
test_model_with_questions(
    dpo_trainer.model, tokenizer, questions,
    system_message=SYSTEM_PROMPT,
    title="Greedy check",
    do_sample=False,                  # <‑ force greedy
)

In [ ]:
test_model_with_questions(
    dpo_trainer.model, tokenizer, questions,
    system_message=SYSTEM_PROMPT,
    title="Greedy check",
    do_sample=True,                  # <‑ force greedy
)

In [ ]:
%pip install huggingface_hub
from huggingface_hub import login
hf_token = user_secrets.get_secret("HUGGING_FACE_TOKEN")

login(token=hf_token)
# Define repo name (should be unique under your namespace)
repo_id = "hjerpe/Qwen2.5-0.5B-Instruct-DPO"

# Push the dataset
# Push to hub
dpo_trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

In [ ]:
%pip install huggingface_hub

In [ ]:
#from huggingface_hub import login
#hf_token = user_secrets.get_secret("HUGGING_FACE_TOKEN")
#
#login(token=hf_token)
## Define repo name (should be unique under your namespace)
#repo_id = "hjerpe/Qwen2.5-0.5B-Instruct-merged-DPO"
#merged_model = model.merge_and_unload()   # returns a **plain** HF model
#
## Push the dataset
## Push to hub
#merged_model.push_to_hub(repo_id)
#tokenizer.push_to_hub(repo_id)

In [ ]:
!pkill jupyter